# Writing your own environment and reward module

<a href="https://colab.research.google.com/github/d-tiapkin/gfnx/blob/main/notebooks/03_custom_environment.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A `gfnx` task is two independent pieces:

- an **environment** (subclass of `gfnx.base.BaseEnvironment`) that describes the
  generation DAG: forward and backward steps, action masks and observations;
- a **reward module** (subclass of `gfnx.base.BaseRewardModule`) that scores terminal
  states.

We build both for a toy task: **sample a subset of size $k$ from $n$ items**, with
probability proportional to $R(S) = \exp\big(\sum_{i \in S} w_i\big)$. Items are added one
at a time in any order, so every subset is reached by $k!$ trajectories and the backward
policy matters. Then we check the environment and train a Trajectory Balance sampler on it.

## Setup

`gfnx` supports Python 3.12–3.14. On Colab, the cell below installs it from GitHub. If
the install upgrades `numpy` or `jax` while an older version is already loaded, it
restarts the runtime once: then run the notebook again from the top (the install is
skipped the second time).

In [ ]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("gfnx") is None:
    if not (3, 12) <= sys.version_info[:2] <= (3, 14):
        raise RuntimeError(f"gfnx supports Python 3.12-3.14, this runtime has {sys.version}.")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "gfnx @ git+https://github.com/d-tiapkin/gfnx.git",
            "equinox",
            "optax",
        ],
        check=True,
    )
    from importlib.metadata import version

    stale = [
        name
        for name in ("numpy", "jax")
        if name in sys.modules and sys.modules[name].__version__ != version(name)
    ]
    if stale and "google.colab" in sys.modules:
        print(f"Upgraded {stale}. Restarting the runtime: run the notebook again from the top.")
        os.kill(os.getpid(), 9)

In [ ]:
import itertools

import chex
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

import gfnx
from gfnx import spaces
from gfnx.base import (
    BaseEnvironment,
    BaseEnvParams,
    BaseEnvState,
    BaseRewardModule,
    BaseRewardParams,
)

## 1. State and parameters

States and parameters are frozen `chex` dataclasses, i.e. JAX pytrees. A state must carry
the three flags of `BaseEnvState`: `is_terminal`, `is_initial` and `is_pad`.

In [ ]:
@chex.dataclass(frozen=True)
class SubsetState(BaseEnvState):
    members: chex.Array  # [n] bool: which items are in the subset
    is_terminal: chex.Array
    is_initial: chex.Array
    is_pad: chex.Array


@chex.dataclass(frozen=True)
class SubsetParams(BaseEnvParams):
    n: int
    k: int

## 2. The environment

Every method acts on a **single** state (no batch dimension). Two conventions keep
rollouts fixed-length under `jax.lax.scan`:

- stepping forward from a terminal state returns the same state with `is_pad=True`
  (and symmetrically for a backward step from the initial state);
- masks use `True` = valid action.

Backward actions describe *structural* moves ("remove item $i$"), and
`get_backward_action` / `get_forward_action` translate between the two directions. Here
they coincide: adding item $i$ is undone by removing item $i$.

In [ ]:
class SubsetEnvironment(BaseEnvironment[SubsetState, SubsetParams]):
    """Build a subset of size k of {0, ..., n - 1} by adding one item per step."""

    def __init__(self, n: int, k: int):
        self.n, self.k = n, k

    def init(self, rng_key):
        return SubsetParams(n=self.n, k=self.k)

    def reset(self):
        return SubsetState(
            members=jnp.zeros(self.n, dtype=jnp.bool_),
            is_terminal=jnp.bool_(False),
            is_initial=jnp.bool_(True),
            is_pad=jnp.bool_(False),
        )

    @property
    def max_steps_in_episode(self):
        return self.k  # every trajectory adds exactly k items

    def step(self, state, action, env_params):
        members = state.members.at[action].set(True)
        next_active = SubsetState(
            members=members,
            is_terminal=members.sum() == self.k,
            is_initial=jnp.bool_(False),
            is_pad=jnp.bool_(False),
        )
        next_state = jax.tree.map(  # freeze terminal states as padding
            lambda pad, active: jnp.where(state.is_terminal, pad, active),
            state.replace(is_pad=True),
            next_active,
        )
        return self.get_obs(next_state, env_params), next_state, next_state.is_terminal, {}

    def backward_step(self, state, backward_action, env_params):
        members = state.members.at[backward_action].set(False)
        prev_active = SubsetState(
            members=members,
            is_terminal=jnp.bool_(False),
            is_initial=members.sum() == 0,
            is_pad=jnp.bool_(False),
        )
        prev_state = jax.tree.map(  # freeze the initial state as padding
            lambda pad, active: jnp.where(state.is_initial, pad, active),
            state.replace(is_pad=True),
            prev_active,
        )
        return self.get_obs(prev_state, env_params), prev_state, prev_state.is_initial, {}

    def get_obs(self, state, env_params):
        return state.members.astype(jnp.float32)

    def get_action_mask(self, state, env_params):
        return ~state.members  # can add any item not yet in the subset

    def get_backward_action_mask(self, state, env_params):
        return state.members  # can remove any item in the subset

    def get_backward_action(self, state, forward_action, next_state, env_params):
        return forward_action

    def get_forward_action(self, state, backward_action, prev_state, env_params):
        return backward_action

    @property
    def name(self):
        return f"Subset-{self.n}-{self.k}-v0"

    @property
    def action_space(self):
        return spaces.Discrete(self.n)

    @property
    def backward_action_space(self):
        return spaces.Discrete(self.n)

    @property
    def observation_space(self):
        return spaces.Box(low=0.0, high=1.0, shape=(self.n,))

    @property
    def state_space(self):
        return spaces.Dict(
            {
                "members": spaces.Box(low=0, high=1, shape=(self.n,), dtype=jnp.bool_),
                "is_terminal": spaces.Box(low=0, high=1, shape=(), dtype=jnp.bool_),
                "is_initial": spaces.Box(low=0, high=1, shape=(), dtype=jnp.bool_),
                "is_pad": spaces.Box(low=0, high=1, shape=(), dtype=jnp.bool_),
            }
        )

## 3. The reward module

The reward knows nothing about the dynamics, only about terminal states. Everything it
needs at call time lives in `reward_params`, created by `init`. Here, the item weights
$w_i$ are sampled once from $\mathcal{N}(0, 1)$.

In [ ]:
@chex.dataclass(frozen=True)
class SubsetRewardParams(BaseRewardParams):
    weights: chex.Array  # [n]


class SubsetRewardModule(BaseRewardModule[SubsetState, SubsetParams, SubsetRewardParams]):
    """R(S) = exp(sum of the weights of the items in S)."""

    def init(self, rng_key, dummy_state):
        n = dummy_state.members.shape[-1]
        return SubsetRewardParams(weights=jax.random.normal(rng_key, (n,)))

    def log_reward(self, state, reward_params):
        return jnp.sum(jnp.where(state.members, reward_params.weights, 0.0))

    def reward(self, state, reward_params):
        return jnp.exp(self.log_reward(state, reward_params))

## 4. Sanity checks

Two properties catch most bugs in a new environment:

1. a **backward step undoes a forward step**:
   `backward_step(step(s, a), get_backward_action(s, a, s'))` returns `s`;
2. **rollouts terminate** within `max_steps_in_episode` and end in valid objects.

In [ ]:
N, K = 10, 4
env = SubsetEnvironment(n=N, k=K)
env_params = env.init(jax.random.PRNGKey(0))
reward_module = SubsetRewardModule()
reward_params = reward_module.init(jax.random.PRNGKey(1), env.reset())


def uniform_policy(rng_key, obs, params):
    logits = jnp.zeros(env.action_space.n)
    return logits, {"forward_logits": logits, "backward_logits": logits}


traj_data, final_states, info = jax.vmap(
    lambda key: gfnx.utils.forward_rollout(key, uniform_policy, None, env, env_params)
)(jax.random.split(jax.random.PRNGKey(2), 512))

# Check 1: invert every real transition.
transitions = jax.vmap(gfnx.utils.split_traj_to_transitions)(traj_data)
transitions = jax.tree.map(lambda x: x.reshape((-1,) + x.shape[2:]), transitions)  # [B*T, ...]
bwd_actions = env.get_backward_action_batch(
    transitions.state, transitions.action, transitions.next_state, env_params
)
_, recovered, _, _ = jax.vmap(env.backward_step, in_axes=(0, 0, None))(
    transitions.next_state, bwd_actions, env_params
)
real = transitions.valid
assert bool(jnp.all((recovered.members == transitions.state.members).all(-1) | ~real))

# Check 2: every rollout ends in a subset of size exactly K.
assert bool(final_states.is_terminal.all())
assert bool((final_states.members.sum(-1) == K).all())
print("checks passed; trajectory length:", float(info["trajectory_length"].mean()))

There are only $\binom{10}{4} = 210$ subsets, so we can enumerate the exact target
distribution for evaluation.

In [ ]:
all_subsets = jnp.array(list(itertools.combinations(range(N), K)))  # [210, K]
all_members = jnp.zeros((len(all_subsets), N), dtype=jnp.bool_)
all_members = all_members.at[jnp.arange(len(all_subsets))[:, None], all_subsets].set(True)
all_states = SubsetState(
    members=all_members,
    is_terminal=jnp.ones(len(all_subsets), dtype=jnp.bool_),
    is_initial=jnp.zeros(len(all_subsets), dtype=jnp.bool_),
    is_pad=jnp.zeros(len(all_subsets), dtype=jnp.bool_),
)
log_rewards_all = jax.vmap(reward_module.log_reward, in_axes=(0, None))(all_states, reward_params)
true_dist = jax.nn.softmax(log_rewards_all)
log_Z = jax.nn.logsumexp(log_rewards_all)
powers = 2 ** jnp.arange(N)  # subset -> integer code, to match samples to subsets
code_to_index = {int(c): i for i, c in enumerate(all_members @ powers)}


def empirical_distribution(states):
    codes = jax.device_get(states.members @ powers)
    counts = jnp.bincount(
        jnp.array([code_to_index[int(c)] for c in codes]), length=len(all_subsets)
    )
    return counts / counts.sum()


tv_uniform = 0.5 * jnp.abs(empirical_distribution(final_states) - true_dist).sum()
print(f"log Z = {float(log_Z):.3f}; TV(uniform policy, target) ≈ {float(tv_uniform):.3f}")

## 5. Train Trajectory Balance on the new environment

Nothing below is specific to the subset task: the policy network, the loss and the
training step are the same as for the built-in environments. This time the network also
outputs **backward logits**, so $P_B$ is learned rather than fixed to uniform.

In [ ]:
class Policy(eqx.Module):
    mlp: eqx.nn.MLP

    def __init__(self, key):
        self.mlp = eqx.nn.MLP(in_size=N, out_size=2 * N, width_size=128, depth=2, key=key)

    def __call__(self, obs):
        forward_logits, backward_logits = jnp.split(self.mlp(obs), 2)
        return {"forward_logits": forward_logits, "backward_logits": backward_logits}


model_params, model_static = eqx.partition(Policy(jax.random.PRNGKey(3)), eqx.is_array)


def policy_fn(rng_key, obs, params):
    out = eqx.combine(params, model_static)(obs)
    return out["forward_logits"], out


def tb_loss(params, traj_data, log_rewards):
    net = eqx.combine(params["model"], model_static)
    traj_data = traj_data.replace(info=jax.vmap(jax.vmap(net))(traj_data.obs))
    log_pf, log_pb = jax.vmap(
        lambda traj: gfnx.utils.forward_trajectory_log_probs(env, traj, env_params)
    )(traj_data)
    return jnp.mean((params["logZ"] + log_pf - log_rewards - log_pb) ** 2)


optimizer = optax.multi_transform(
    {"net": optax.adam(1e-3), "logZ": optax.adam(1e-1)},
    {"model": "net", "logZ": "logZ"},
)
params = {"model": model_params, "logZ": jnp.array(0.0)}
opt_state = optimizer.init(params)


def train_step(carry, key):
    params, opt_state = carry
    traj_data, final_states, _ = jax.vmap(
        lambda k: gfnx.utils.forward_rollout(k, policy_fn, params["model"], env, env_params)
    )(jax.random.split(key, 64))
    log_rewards = jax.vmap(reward_module.log_reward, in_axes=(0, None))(
        final_states, reward_params
    )
    loss, grads = jax.value_and_grad(tb_loss)(params, traj_data, log_rewards)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    return (optax.apply_updates(params, updates), opt_state), loss


@jax.jit
def train(params, opt_state, key):
    (params, opt_state), losses = jax.lax.scan(
        train_step, (params, opt_state), jax.random.split(key, 500)
    )
    return params, opt_state, losses


@jax.jit
def sample(model_params, key):
    _, final_states, _ = jax.vmap(
        lambda k: gfnx.utils.forward_rollout(k, policy_fn, model_params, env, env_params)
    )(jax.random.split(key, 20_000))
    return final_states


key = jax.random.PRNGKey(4)
for epoch in range(4):
    key, train_key, sample_key = jax.random.split(key, 3)
    params, opt_state, losses = train(params, opt_state, train_key)
    tv = (
        0.5
        * jnp.abs(empirical_distribution(sample(params["model"], sample_key)) - true_dist).sum()
    )
    print(
        f"step {(epoch + 1) * 500:5d} | loss {float(losses[-50:].mean()):.4f} | "
        f"logZ {float(params['logZ']):.3f} (true {float(log_Z):.3f}) | TV ≈ {float(tv):.3f}"
    )

The TV estimate is computed from 20,000 samples, so even a perfect sampler scores a few
hundredths. The learned and target probabilities of the 210 subsets:

In [ ]:
learned = empirical_distribution(sample(params["model"], jax.random.PRNGKey(5)))
plt.scatter(true_dist, learned, s=10)
lim = float(true_dist.max()) * 1.1
plt.plot([0, lim], [0, lim], "k--", linewidth=1)
plt.xlabel("target probability R(S) / Z")
plt.ylabel("sampling frequency")
plt.show()

## Going further

- Make the environment **enumerable** by implementing `is_enumerable`, `get_all_states`,
  `state_to_index`, `get_true_distribution`, `get_empirical_distribution`, ... Then
  `gfnx.metrics.ExactDistributionMetricsModule` can evaluate it exactly, as in the
  quickstart notebook.
- Everything in this notebook is a pure function of arrays, so the vmap-over-seeds recipe
  of notebook 2 applies unchanged.
- The built-in environments in `src/gfnx/environment/` are short and follow exactly this
  structure; their docstrings are rendered in the
  [API reference](https://gfnx.readthedocs.io/en/latest/api/).